<a href="https://colab.research.google.com/github/hussain-azmat/colab_work/blob/main/ONNX.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install onnxruntime opencv-python matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 73.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 6.0 MB/s eta 0:00:00


In [ ]:
import os
import urllib.request

# URLs for the ONNX models and model.txt
face_detector_url = "https://github.com/yinguobing/head-pose-estimation/raw/master/assets/face_detector.onnx"
face_landmarks_url = "https://github.com/yinguobing/head-pose-estimation/raw/master/assets/face_landmarks.onnx"
model_txt_url = "https://github.com/yinguobing/head-pose-estimation/raw/master/assets/model.txt"

# Paths to save the files
face_detector_path = "face_detector.onnx"
face_landmarks_path = "face_landmarks.onnx"
model_txt_path = "model.txt"

# Download the files
if not os.path.exists(face_detector_path):
    urllib.request.urlretrieve(face_detector_url, face_detector_path)
    print("Face detector model downloaded successfully!")
else:
    print("Face detector model already exists.")

if not os.path.exists(face_landmarks_path):
    urllib.request.urlretrieve(face_landmarks_url, face_landmarks_path)
    print("Face landmarks model downloaded successfully!")
else:
    print("Face landmarks model already exists.")

if not os.path.exists(model_txt_path):
    urllib.request.urlretrieve(model_txt_url, model_txt_path)
    print("model.txt downloaded successfully!")
else:
    print("model.txt already exists.")

# Read and print the contents of model.txt
with open(model_txt_path, "r") as file:
    contents = file.read()
    print(contents)

Face detector model already exists.
Face landmarks model already exists.
model.txt already exists.
-73.393523 
-72.775014 
-70.533638 
-66.850058 
-59.790187 
-48.368973 
-34.121101 
-17.875411 
0.098749 
17.477031 
32.648966 
46.372358 
57.343480 
64.388482 
68.212038 
70.486405 
71.375822 
-61.119406 
-51.287588 
-37.804800 
-24.022754 
-11.635713 
12.056636 
25.106256 
38.338588 
51.191007 
60.053851 
0.653940 
0.804809 
0.992204 
1.226783 
-14.772472 
-7.180239 
0.555920 
8.272499 
15.214351 
-46.047290 
-37.674688 
-27.883856 
-19.648268 
-28.272965 
-38.082418 
19.265868 
27.894191 
37.437529 
45.170805 
38.196454 
28.764989 
-28.916267 
-17.533194 
-6.684590 
0.381001 
8.375443 
18.876618 
28.794412 
19.057574 
8.956375 
0.381549 
-7.428895 
-18.160634 
-24.377490 
-6.897633 
0.340663 
8.444722 
24.474473 
8.449166 
0.205322 
-7.198266 
-29.801432 
-10.949766 
7.929818 
26.074280 
42.564390 
56.481080 
67.246992 
75.056892 
77.061286 
74.758448 
66.929021 
56.311389 
42.419126 


In [ ]:
import cv2
import numpy as np
import onnxruntime as ort

# Load the ONNX models
face_detector_session = ort.InferenceSession(face_detector_path)
face_landmarks_session = ort.InferenceSession(face_landmarks_path)

# Define input and output names
face_detector_input_name = face_detector_session.get_inputs()[0].name
face_detector_output_name = face_detector_session.get_outputs()[0].name
face_landmarks_input_name = face_landmarks_session.get_inputs()[0].name
face_landmarks_output_name = face_landmarks_session.get_outputs()[0].name

# Helper function to preprocess the image for face detection
def preprocess_face_detection(image):
    # Resize the image to the model's input size (320x320)
    resized_image = cv2.resize(image, (320, 320))
    # Convert to RGB (if needed)
    rgb_image = cv2.cvtColor(resized_image, cv2.COLOR_BGR2RGB)
    # Normalize the image to [-1, 1]
    normalized_image = (rgb_image.astype(np.float32) - 127.5) / 128.0
    # Transpose to match the model's input format (1, 3, 320, 320)
    input_data = np.transpose(normalized_image, (2, 0, 1))
    input_data = np.expand_dims(input_data, axis=0)
    return input_data

# Helper function to preprocess the face region for landmark detection
def preprocess_face_landmarks(face_region):
    # Resize the face region to the model's input size (64x64)
    resized_face = cv2.resize(face_region, (64, 64))
    # Convert to RGB (if needed)
    rgb_face = cv2.cvtColor(resized_face, cv2.COLOR_BGR2RGB)
    # Normalize the image to [0, 1]
    normalized_face = rgb_face.astype(np.float32) / 255.0
    # Transpose to match the model's input format (1, 3, 64, 64)
    input_data = np.transpose(normalized_face, (2, 0, 1))
    input_data = np.expand_dims(input_data, axis=0)
    return input_data

# Helper function to postprocess the face detection output
def postprocess_face_detection(output, image_shape):
    # Example: Assume the output is a single array with shape [1, N, 7]
    # Where N is the number of detected faces, and each face has 7 values:
    # [batch_id, class_id, confidence, x1, y1, x2, y2]
    detections = output[0]  # Shape: [1, N, 7]

    # Filter out low-confidence detections
    confidence_threshold = 0.5
    valid_detections = detections[detections[:, :, 2] > confidence_threshold]

    # Extract bounding boxes, confidence scores, and class IDs
    boxes = valid_detections[:, 3:7]  # [x1, y1, x2, y2]
    scores = valid_detections[:, 2]   # Confidence scores
    class_ids = valid_detections[:, 1]  # Class IDs

    # Rescale bounding boxes to the original image size
    height, width = image_shape[:2]
    boxes[:, [0, 2]] *= width
    boxes[:, [1, 3]] *= height
    boxes = boxes.astype(np.int32)

    return boxes, scores, class_ids

# Helper function to postprocess the face landmarks output
def postprocess_face_landmarks(output):
    # The output is a set of 68 facial landmarks
    landmarks = output[0].reshape(-1, 2)
    return landmarks

In [ ]:
# Load the video
video_path = "test4.mp4"  # Replace with your video path
cap = cv2.VideoCapture(video_path)

# Check if the video opened successfully
if not cap.isOpened():
    print("Error: Could not open video.")
    exit()

# Process the video frame by frame
while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Preprocess the frame for face detection
    input_data = preprocess_face_detection(frame)

    # Run face detection
    face_detector_output = face_detector_session.run([face_detector_output_name], {face_detector_input_name: input_data})

    print("Face detector output:", face_detector_output)

    # Postprocess the face detection output
    boxes, scores, class_ids = postprocess_face_detection(face_detector_output, frame.shape)

    # Draw bounding boxes and landmarks for each detected face
    for box in boxes:
        x1, y1, x2, y2 = box
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

        # Extract the face region
        face_region = frame[y1:y2, x1:x2]

        # Preprocess the face region for landmark detection
        input_data = preprocess_face_landmarks(face_region)

        # Run landmark detection
        face_landmarks_output = face_landmarks_session.run([face_landmarks_output_name], {face_landmarks_input_name: input_data})

        # Postprocess the landmarks
        landmarks = postprocess_face_landmarks(face_landmarks_output)

        # Draw landmarks on the face region
        for (x, y) in landmarks:
            x = int(x * (x2 - x1) + x1)
            y = int(y * (y2 - y1) + y1)
            cv2.circle(frame, (x, y), 2, (0, 0, 255), -1)

    # Display the frame
    cv2.imshow("Video", frame)

    # Exit if 'q' is pressed
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Release the video capture object and close windows
cap.release()
cv2.destroyAllWindows()

Face detector output: [array([[0.02107647],
       [0.02617338],
       [0.00910121],
       ...,
       [0.00907269],
       [0.01437026],
       [0.0210765 ]], dtype=float32)]


IndexError: too many indices for array: array is 2-dimensional, but 3 were indexed